### loading the RE² dataset and first diagnose
because the papers.zip is separate from the main dataset i have to create a column paper text separately


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Loading RE² Dataset
from datasets import load_dataset
import json
import zipfile
from collections import Counter

ds = load_dataset("Daoze/ReviewRebuttal",
                  data_files="REVIEWS_train.json",
                  split="train",
                  cache_dir='./my_dataset')
print(ds.column_names)
print(ds[0])

# slicing 500 samples
sample = ds.select(range(min(500, len(ds))))
print(f"{len(sample)} samples loaded")

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

REVIEWS_train.json: reconstructing file:   0%|          |  0.00B /  319MB            

REVIEWS_train.json: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

['paper_id', 'conference_year_track', 'reviews', 'review_initial_ratings_unified', 'review_final_ratings_unified', 'metareview', 'decision']
{'paper_id': '00I1UhSk55i', 'conference_year_track': 'AAAI 2022 Workshop ADAM', 'reviews': [{'reviewer_id': 'AnonReviewer', 'review_title': 'Well written paper on RL for TO', 'review_content': 'This paper presents a RL approach for solving TO problems in 2D. A sensible framing of the state space, action space is presented. By formulating the transition probability in terms of convolutional operators, the authors are able to smartly utilize a lower resolution FE solver to learn the transition probability. This along with progressive refinement results in a very competitive approach.\n\nA couple of clarifications\n\n- Can the authors show a result with different locations of the 2 bounded elements? While the authors claim that the bounded elements are randomly assigned, every result shown only has bounded elements on the two edges of the domain. In 

In [ ]:
# checking unique ids
ids = {r['paper_id'] for r in sample}
print(f"{len(ids)} unique ids")

500 unique ids


In [ ]:
# Loading and scanning papers.zip without extracting everything
from huggingface_hub import hf_hub_download
local_zip_path = hf_hub_download(
    repo_id="Daoze/ReviewRebuttal",
    filename="papers.zip",
    repo_type="dataset"
)
print(f"Downloaded to {local_zip_path}")



papers.zip: reconstructing file:   0%|          |  0.00B / 12.4GB            

papers.zip: downloading bytes:           |  0.00B            

Downloaded to /root/.cache/huggingface/hub/datasets--Daoze--ReviewRebuttal/snapshots/0171bacee24a9d405792a32231dabcdb3788b365/papers.zip


In [ ]:
# checking total papers and if any matches the paper id from the original dataset
zf = zipfile.ZipFile(f"{local_zip_path}", "r")
all_paths = zf.namelist()
print(f"papers.zip contains {len(all_paths)} files total")

papers.zip contains 373425 files total


In [ ]:
id_to_paths = {}
for path in all_paths:
  for pid in ids:
    if f"/{pid}/" in path:
      id_to_paths.setdefault(pid, []).append(path)

In [ ]:
print(len(id_to_paths.keys()))

500


In [ ]:
# Diagnozing match rate
matched_ids = set(id_to_paths.keys())
unmatched_ids = ids - matched_ids
print(f"Matched {len(matched_ids)}/{len(ids)} paper ids "
      f"{len(matched_ids)/len(ids):.1%}")

Matched 500/500 paper ids 100.0%


In [ ]:
# counting format availability
fmta_counts = Counter()
for pid, path in id_to_paths.items():
  has_tex = any(p.endswith(".tex") for p in path)
  has_md = any(p.endswith(".md") for p in path)
  fmta_counts[(has_tex, has_md)] += 1
print("has_tex, has_md, counted", dict(fmta_counts))

has_tex, has_md, counted {(True, True): 500}


In [ ]:
# Extracting matched files
paper_text ={}
extraction_errors=[]
for pid, path in id_to_paths.items():
  tex_paths = [p for p in path if p.endswith(".tex")]
  md_paths =  [p for p in path if p.endswith(".md")]
  chosen = md_paths[0] if md_paths else (tex_paths[0] if tex_paths else None)
  if chosen is None:
    continue
  try:
    with zf.open(chosen) as fh:
      paper_text[pid] = fh.read().decode("utf-8", errors = "replace")
  except Exception as e:
    extraction_errors.append((pid, chosen, str(e)))
print(f"Extracted text for {len(paper_text)} paper_ids")
if extraction_errors:
  print(f"{len(extraction_errors)}Extraction errors for inspection ")
print(extraction_errors[:5])

Extracted text for 500 paper_ids
[]


In [ ]:
# Sanity Check for Text length
lengths = [len(t) for t in paper_text.values()]
if lengths:
  print(f"\n min char lenth {min(lengths)} \n max char length {max(lengths)} \n avg char length {sum(lengths)/len(lengths)}")
print(f"rought token estimate (4 char = 1 token) : {(sum(lengths)/len(lengths))/4}")



 min char lenth 9431 
 max char length 185566 
 avg char length 45548.474
rought token estimate (4 char = 1 token) : 11387.1185


In [ ]:
import re

# Broader markers for both references and appendix — covers numbered
# headings, bold-only lines, and common LaTeX bibliography commands
# your first pass missed (printbibliography, bibliographystyle, \appendix).
REF_PATTERN = re.compile(
    r'(\\bibliography\{|\\bibliographystyle\{|\\printbibliography|'
    r'\\begin\{thebibliography\}|'
    r'^#{1,4}\s*(?:[\dIVXLC]+\.?\s*)?references?\s*$|'
    r'^\*{0,2}\s*references?\s*\*{0,2}\s*$)',
    re.IGNORECASE | re.MULTILINE
)

APPENDIX_PATTERN = re.compile(
    r'(\\appendix\b|'
    r'^#{1,4}\s*(?:[\dIVXLC]+\.?\s*)?appendix(es)?\s*$|'
    r'^\*{0,2}\s*appendix(es)?\s*\*{0,2}\s*$)',
    re.IGNORECASE | re.MULTILINE
)

def strip_refs_and_appendix(text):
    ref_match = REF_PATTERN.search(text)
    app_match = APPENDIX_PATTERN.search(text)
    cut_points = [m.start() for m in (ref_match, app_match) if m]
    return text[:min(cut_points)] if cut_points else text

before_lens, after_lens = [], []
no_match_ids = []
for pid, text in paper_text.items():
    stripped = strip_refs_and_appendix(text)
    before_lens.append(len(text))
    after_lens.append(len(stripped))
    if len(stripped) == len(text):
        no_match_ids.append(pid)

avg_before = sum(before_lens) / len(before_lens)
avg_after = sum(after_lens) / len(after_lens)
print(f"avg chars before: {avg_before:.0f}")
print(f"avg chars after:  {avg_after:.0f}")
print(f"avg reduction:    {(1 - avg_after/avg_before):.1%}")
print(f"max after:        {max(after_lens)}")
print(f"no match: {len(no_match_ids)}/{len(paper_text)}")

# Don't trust the aggregate — actually look at why these failed
for pid in no_match_ids[:3]:
    print(f"\n--- {pid} tail (last 400 chars) ---")
    print(paper_text[pid][-400:])

avg chars before: 45548
avg chars after:  33940
avg reduction:    25.5%
max after:        91363
no match: 2/500

--- S6Pl8ztg_b5 tail (last 400 chars) ---
es. We report the average performance of five different random seeds and standard deviation (in brackets). "-gold" indicates that the models are trained and tested with instances with gold knowledge. We cannot directly compare them with the normal setting, but it could serve as the upper-bound for our learning paradigm. Best performing models under both settings are indicated with the bold font.



--- JZFg3r98XMV tail (last 400 chars) ---
epochs of training period, in order to find if we overfit the data or on the contrary. But we regretfully found that this doesn't lead to a better performance for PPO agent, no matter how we change the number of epochs, the curve just goes downwards. So in our circumstance, we have drawn the conclusion that PG is more suitable than PPO method to train the agent to make a deal with American stocks.


In [ ]:
## Creating a separate column
# Convert to a standard list of dicts so in-place updates are kept
sample_list = list(sample)

for r in sample_list:

  pid = r["paper_id"]
  # def strip_refs_and_appendix(paper_text) to do !
  r["paper_text"] = paper_text.get(pid)
  r["match_status"] = "matched" if pid in paper_text else "unmatched"

status_counts = Counter(r["match_status"] for r in sample_list)
print(f"final status on {len(sample_list)} records : {dict(status_counts)}")

# Fix: Serialize the list of dictionaries (sample_list) instead of the Dataset object (sample)
with open("reviews_sample_joined.json", "w", encoding="utf-8") as f:
  json.dump(sample_list, f, indent = 2)
print("wrote reviews_sample_joined.json (pending for inspection)")

final status on 500 records : {'matched': 500}
wrote reviews_sample_joined.json (pending for inspection)


### Step 1: Installing a Training Stack
Going to be the same as our teaching notebook

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q --upgrade "transformers>=4.51.0" "peft>=0.17.0" "accelerate>=1.7.0" "bitsandbytes>=0.46.0" "datasets>=3.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 133.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


In [ ]:
import hashlib
import platform
from importlib.metadata import version
from packaging.version import Version

# import bitsandbytes
import numpy as np
import torch
import transformers
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,

)

for package in ("transformers", "peft", "accelerate",  "datasets"):
  print(f"{package}: {version(package)}")
print(f"Python Version {platform.python_version()}")
print(f"PyTorch Version {torch.__version__}")
print(f"GPU Mem: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

transformers: 5.19.0
peft: 0.21.2
accelerate: 1.15.0
datasets: 5.1.0
Python Version 3.13.15
PyTorch Version 2.11.0+cu130
GPU Mem: 79.3 GB


### Step 2: Defnining a T4 Safe Profile
Going to be the same as tutorial notebook except: MODEL_ID, and MAX_ROWS

In [ ]:
# Defining Constants
SEED = 42
MODEL_ID = "Qwen/Qwen3-1.7B"
MAX_LENGTH = 6144
TRAIN_BATCH_SIZE = 5
EVAL_BATCH_SIZE = 5
GRADIENT_ACCUMULATION_STEPS = 10
LEARNING_RATE = 2e-4

MAX_TRAIN_ROWS = 1200
MAX_EVAL_ROWS = 120
MAX_TEST_ROWS = 120
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:

# Tokenizer Setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Bits and Bytes Configuration

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = quantization_config,
    device_map = "auto", #: run as a plain single-process script, and don't launch with accelerate launch --multi_gpu or torchrun, because those conflict with device_map="auto"'s own placement logic
    dtype=torch.bfloat16, # torch_dtype is deprecated
    low_cpu_mem_usage=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()
print(f"Loaded Model Footprint (1.7B): {model.get_memory_footprint() / 2**30:.2f} GB")



config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded Model Footprint (1.7B): 1.24 GB


### Step 3: Load and Split the Review data
Raw Dataset Different, Required Columns Different, rng shuffle might be needed, Take Whole papers may be needed, splits are required and assertions are recommended

In [ ]:
import random

# stripping references and appendix sections to save context cost
paper_text = {
    pid: strip_refs_and_appendix(text)
    for pid, text in paper_text.items()
}

for r in sample_list:
  pid = r["paper_id"]
  r["paper_text"] = paper_text.get(pid)
  r["match_status"] = "matched" if pid in paper_text else "unmatched"

# Loading the dataset and checking if the required columns for the training are present
raw_dataset = load_dataset("json", data_files="/content/reviews_sample_joined.json", split='train')
raw_dataset = raw_dataset.filter(lambda r: r["match_status"] == "matched")
print(f"{len(raw_dataset)} matched records after filtering")

required_columns = {"paper_id", "paper_text", "reviews"}
assert required_columns.issubset(raw_dataset.column_names), raw_dataset.column_names

# Creating Groups by paper-> review
rows = []
for example in raw_dataset:
  paper_id = example["paper_id"]
  paper_text_val = example["paper_text"]

  for i, review in enumerate(example["reviews"]):
    rows.append({
        'unit_id': f"{paper_id}_{i}",
        'paper_id': paper_id,  # changed
        'whole_paper': paper_text_val,
        'review_content': review["review_content"],
        'meta': {
            'review_title': review['review_title'],
            'initial_score': review["initial_score"],
            'final_score': review["final_score"],
            'initial_score_unified': review["initial_score_unified"],
            'final_score_unified': review["final_score_unified"],
        }
    })

# Build dataset from the list of rows
dataset = Dataset.from_list(rows)
dataset = dataset.filter(lambda r: len((r["review_content"] or "").strip()) >= 100) #changed
print(f"{len(dataset)}: paper, review rows built")

# Properly shuffle and split the HF Dataset
# shuffled_dataset = dataset.shuffle(seed=SEED)

# total_len = len(shuffled_dataset)
# train_end = min(MAX_TRAIN_ROWS, total_len)
# val_end = min(train_end + MAX_EVAL_ROWS, total_len)
# test_end = min(val_end + MAX_TEST_ROWS, total_len)

# train_dataset = shuffled_dataset.select(range(0, train_end))
# val_dataset = shuffled_dataset.select(range(train_end, val_end))
# test_dataset = shuffled_dataset.select(range(val_end, test_end))

pids = sorted({r["paper_id"] for r in rows}); np.random.default_rng(SEED).shuffle(pids); n = len(pids)
part = {p: ("train" if i< .8*n else "val" if i < .9*n else "test") for i, p in enumerate(pids)}

def take(name, cap):
  data = dataset.filter(lambda r: part[r["paper_id"]] == name)
  data = data.shuffle(seed=SEED)
  return data.select(range(min(cap, len(data))))

train_dataset, val_dataset, test_dataset = take("train", MAX_TRAIN_ROWS), take("val", MAX_EVAL_ROWS), take("test", MAX_TEST_ROWS)
raw_test_dataset = test_dataset

print(f"train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

# Check for leakage using unit_id sets
train_ids = set(train_dataset['unit_id'])
val_ids = set(val_dataset['unit_id'])
test_ids = set(test_dataset['unit_id'])

pid = lambda ds: {u.rsplit("_", 1)[0] for u in ds["unit_id"]}
print(len(pid(train_dataset) & pid(test_dataset)), "of", len(pid(test_dataset)), "test papers are also in train")

assert len(train_ids & val_ids) == 0, "Leakage found between train and val!"
assert len(train_ids & test_ids) == 0, "Leakage found between train and test!"
assert len(val_ids & test_ids) == 0, "Leakage found between val and test!"
print("No leakage found between splits!")

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

500 matched records after filtering


Filter:   0%|          | 0/1588 [00:00<?, ? examples/s]

1584: paper, review rows built


Filter:   0%|          | 0/1584 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1584 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1584 [00:00<?, ? examples/s]

train: 1200, val: 120, test: 120
0 of 50 test papers are also in train
No leakage found between splits!


### Step 4: Render Prompts and Mask The Labels
Constants Same, Compact text Same, Prompt Text a bit different, completion for a bit different, tokenize with prompt mask same, from here almost all code of step 4 is same


In [ ]:
MAX_PAPER_TEXT_CHARS = 16000
MAX_REVIEW_CHARS = 16000

# cheap cut
# def compact_text(value, max_chars):
#   text = " ".join((value or "").split())
#   if len(text) <= max_chars:
#     return text
#   shortened = text[:max_chars].rsplit(" ", 1)[0].rstrip()
#   return (text[:max_chars] or shortened) + "..."

def compact_text(value, max_chars):
  lines = (value or "").splitlines()
  # Fix: join the words with space first, then join the lines with newline
  text = "\n".join(" ".join(line.split()) for line in lines).strip()
  while "\n\n\n" in text:
    text = text.replace("\n\n\n", "\n") # Fix: reassign the replaced string
  if len(text)> max_chars:
    text = text[:max_chars].rsplit(" ", 1)[0] + "..."
  return text

def prompt_for(example):
  abstract = compact_text(example["whole_paper"], MAX_PAPER_TEXT_CHARS)
  user_text = f"write a constructive peer review for this paper. \n\n{abstract}"
  messages = [{'role': 'user', 'content': user_text}]

  return tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False,
  )

def completion_for(example):
  return compact_text(example["review_content"], MAX_REVIEW_CHARS) + tokenizer.eos_token

def tokenize_with_prompt_mask(example):
  prompt = prompt_for(example)
  full_text = prompt + completion_for(example)
  # with_meta = prompt + example['review_content'] + example['meta']
  full_ids = tokenizer(full_text, add_special_tokens=False)['input_ids']
  encoded = tokenizer(
      full_text,
      add_special_tokens=False,
      truncation=True,
      max_length=MAX_LENGTH,
  )

  prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
  prompt_length = min(len(prompt_ids), len(encoded['input_ids']))

  assert prompt_ids[:prompt_length] == encoded['input_ids'][:prompt_length], "prompt token boundary mismatch, don't train with misaligned mask"

  labels = list(encoded['input_ids'])
  labels[:prompt_length] = [-100] * prompt_length
  encoded['labels'] = labels
  encoded['untruncated_length'] = len(full_ids)
  encoded['target_tokens'] = sum(token != -100 for token in labels)
  return encoded


def make_tokenized(rows):
  tokenized = rows.map(
      tokenize_with_prompt_mask,
      remove_columns=rows.column_names,
      desc="Tokenizing and using Prompt Mask"
  )
  assert all(count > 0 for count in tokenized['target_tokens']), 'a row lost its whole target '
  return tokenized

train_with_stats = make_tokenized(train_dataset)
val_with_stats = make_tokenized(val_dataset)
test_with_stats = make_tokenized(test_dataset)

for name, split in {
    'train': train_with_stats,
    'val': val_with_stats,
    'test': test_with_stats
}.items():
    lengths = np.asarray(split['untruncated_length'])
    truncated = int(np.sum(lengths > MAX_LENGTH))
    print(f"{name}: \n p50 = {np.percentile(lengths, 50):.0f}\n p90 = {np.percentile(lengths, 90):.0f}\n p10 = {np.percentile(lengths, 10):.0f}\n p100 = {np.percentile(lengths, 100):.0f}\n truncated = {truncated}/{len(split)}")

columns_to_drop = ['untruncated_length', 'target_tokens']
train_dataset = train_with_stats.remove_columns(columns_to_drop)
val_dataset = val_with_stats.remove_columns(columns_to_drop)
test_dataset = test_with_stats.remove_columns(columns_to_drop)

Tokenizing and using Prompt Mask:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [ ]:
# This is for inspection
example = train_dataset[0]
visible_text = tokenizer.decode(example['input_ids'], skip_special_tokens=False)
masked_count = sum (label == -100 for label in example['labels'])
target_count = sum (label != -100 for label in example['labels'])
print(visible_text[:1000])
print(f"tokens {len(example['input_ids'])}  prompt-masked {masked_count} review target {target_count}")

# collator pads -100 values so it never contributes to loss
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors= "pt",
    label_pad_token_id = -100,
)
example_batch = data_collator([train_dataset[0], train_dataset[1]])
print(f"shape {[tuple(tensor.shape) for name, tensor in example_batch.items()]}")


### Step 5: Attaching LoRA Adapters

In [ ]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,},
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Step 6: Training with Hugging

In [27]:
# # 1. Mount Google Drive so you can save files directly to it
# from google.colab import drive
# drive.mount('/content/drive')

# 2. Define a path inside your Drive for the checkpoints
drive_output_dir = "/content/drive/MyDrive/rc_project_LoRA_runs"




In [28]:
training_args = TrainingArguments(
    output_dir = drive_output_dir,
    per_device_train_batch_size = TRAIN_BATCH_SIZE,
    per_device_eval_batch_size = EVAL_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    learning_rate = LEARNING_RATE,
    num_train_epochs = 1,
    fp16 = False,
    bf16 = True,
    optim = "paged_adamw_8bit",
    logging_steps=10,
    logging_first_step = True,
    save_strategy = "steps",
    save_steps = 20,
    save_total_limit = 10,
    report_to = "none",
    seed=SEED,
    dataloader_num_workers = 0,

)

trainer = Trainer(
    model= model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    data_collator = data_collator,

)

torch.cuda.empty_cache() # just to make some space
# train_result = trainer.train(resume_from_checkpoint="/content/drive/MyDrive/rc_project_LoRA_runs/checkpoint-20")
train_result = trainer.train()

print(train_result.metrics)
print("Validation metrics: ", trainer.evaluate(metric_key_prefix="Validation"))
print("Held out Reviews metrics: ", trainer.evaluate(test_dataset, metric_key_prefix="Test"))


Step,Training Loss
1,3.190523
10,2.723941


Step,Training Loss
1,3.190523
10,2.723941
20,2.622556


{'train_runtime': 1371.5724, 'train_samples_per_second': 0.875, 'train_steps_per_second': 0.017, 'total_flos': 5.232954249904128e+16, 'train_loss': 2.669832011063894, 'epoch': 1.0}


Training Loss,Validation Loss,Step
2.622556,2.534798,24


Validation metrics:  {'Validation_loss': 2.5347976684570312}


Training Loss,Validation Loss,Step
2.622556,2.606928,24


Held out Reviews metrics:  {'Test_loss': 2.6069281101226807}


### Step 7 Generating Held out Reviews


In [30]:
model.eval() # switching model to eval mode to remove training only arguments like dropout
model.config.use_cache= True # to enable attention cache

MAX_TITLE_CHARACTERS = 400
MAX_REVIEW_CHARACTERS = 16000

def generate_reviews(example):
  prompt = prompt_for(example)
  inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")
  input_device = next(model.parameters()).device
  inputs = {name: tensor.to(input_device) for name, tensor in inputs.items()}

  with torch.inference_mode():
    generated = model.generate(
        **inputs,
        # max_new_tokens=max_new_tokens,
        do_sample = True,
        temperature = 0.6,
        top_p = 0.8,
        pad_token_id = tokenizer.eos_token_id,

    )
    new_tokens = generated[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True ).strip()

# Retrieve un-tokenized raw samples from shuffled_dataset for the test split


for index, example in enumerate(list(raw_test_dataset)[:3], start=1):
  print(f"{10 * '='} Held out paper no. {index} {'=' * 10}")
  title_val = example["meta"].get("review_title", "No Title Available") if isinstance(example.get("meta"), dict) else "No Title Available"
  print("Title:", compact_text(title_val, MAX_TITLE_CHARACTERS))
  print("Model Review: \n", generate_reviews(example))
  print("Reference Review: \n", compact_text(example["review_content"], MAX_REVIEW_CHARACTERS))

========== Held out paper no. 1 ==========
Title: null


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1955: UserWarning: Using the model-agnostic default `max_length` (=4441) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Model Review: 
 ### Review Summary

This paper proposes a novel closed-loop policy learning method for 6D robotic gras
Reference Review: 
 summary: The paper proposes a method for learning closed-loop 6D robot grasping policies of singulated objects, using segmented aggregated point clouds as input. It proposes a combination of behavior cloning from expert demonstrations and reinforcement learning based on an actor-critic approach. Furthermore, it introduces an auxiliary task for supervised training of the actor and critic networks, which consist of predicting the final grasping pose of the end-effector. During the behavior cloning phase, additional demonstrations are gathered following the DAGGER scheme, where the expert provides the labels for actions and goals. During the RL phase, goal labels are generated either from a set of pre-labeled goals (in case of known objects with annotations) or hind-sight goals are used (in case of unknown objects) to add a goal label to the data tuple

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1955: UserWarning: Using the model-agnostic default `max_length` (=3533) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Model Review: 
 ### Review Summary

This paper presents a novel approach for biomedical relation extraction (RE) using abstractified
Reference Review: 
 Summary
This papers studies distantly supervised biomedical RE, where the data have a long-tail distribution.
The authors hypothesize that formulating bags using distinct entity pairs results in many bags with too few sentences, which may hinder performance.

The authors then propose abstractified multi-instance learning (AMIL) that formulates bags using UMLS semantic type (i.e., biomedical named entity type). AMIL outperforms the previous state-of-the-art method in Amin et al. 2020. Most of the performance gain comes from "rare" tuples, validating the hypothesis in the beginning. The author also empirically compared 17 different methods to construct the representation, where they find the type L achieves the best performance.

Strength
- The proposed method AMIL is a intuitive and effective extension to MIL targeting long-tail data di

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1955: UserWarning: Using the model-agnostic default `max_length` (=3549) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Model Review: 
 ### Reviewer 1

This paper proposes a metamorphic adversarial detection approach that is trained on
Reference Review: 
 This paper proposes to use statistical techniques to learn optimal thresholds of separation between clean and adversarial examples, and achieves excellent detection accuracies.

Experimental performance can be demonstrated that the method can effectively detect adversarial examples and achieve a good performance. However, the weakness is also listed as follows:

1. More datasets should be introduced for comprehensive comparisons, such as MS-Celeb-1M[1].

2. Will different face detectors affect the final performance?

[1] MS-Celeb-1M: A Dataset and Benchmark for Large-Scale Face Recognition


In [31]:
import shutil

# Zip the checkpoint folder
shutil.make_archive(
    base_name='/content/checkpoint-24_archive',
    format='zip',
    root_dir='/content/drive/MyDrive/rc_project_LoRA_runs/checkpoint-24'
)
print("Archive created successfully at: /content/checkpoint-24_archive.zip")

Archive created successfully at: /content/checkpoint-24_archive.zip


In [34]:
ADAPTER_DIR = '/content/rc_project_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("save adapter and tokenizer to: ", ADAPTER_DIR)
import shutil

# Zip the checkpoint folder
shutil.make_archive(
    base_name='/content/rc_project_adapter_v1',
    format='zip',
    root_dir='/content/rc_project_adapter'
)
print("Archive created successfully at: /content/rc_project_adapter_v1")

save adapter and tokenizer to:  /content/rc_project_adapter
Archive created successfully at: /content/rc_project_adapter_v1
